# Chapter 2 — AI Applications: Image, Video, Speech & Real-World Domains

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part1_foundations/ch02_ai_applications/ch02_notebook.ipynb)

Companion notebook for **Chapter 2** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Image tasks include classification, clustering, segmentation, retrieval, captioning, VQA and generation.
- Image retrieval can work by example (needs a similarity measure) or by keyword (needs annotated images).
- Video is multimodal: frames, audio and text all carry information.
- Speech tasks cover recognizing words, speakers, languages and emotions, plus synthesis and translation.
- Recognizing who or what is in audio should not depend on the words spoken, the language or the channel.
- Banking and healthcare are major users of AI: fraud detection, credit scoring, diagnosis and personalized treatment.

## Example 1: Image Classification on Handwritten Digits

Scikit-learn includes 1,797 tiny 8x8 images of handwritten digits. Each image is 64 pixel values, so to the computer an image is just a list of 64 numbers. The program draws one digit as text so you can see it, trains a support vector machine (you will meet SVMs in Chapters 11 and 12) on 70% of the images, and tests on the rest. It reaches 99.4% accuracy and reports which digits it confused.

In [1]:
# Chapter 2 - Example 1: Image classification on handwritten digits
# Each 8x8 image becomes 64 numbers (pixel intensities); a classifier learns the digits.
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

digits = load_digits()                      # 1,797 images, built into scikit-learn
X, y = digits.data, digits.target
print("Images:", X.shape[0], "| pixels per image:", X.shape[1], "| classes:", sorted(set(y.tolist())))

# Show one image as text so we can 'see' it
img = digits.images[0]
print(f"\nImage 0 (label {y[0]}):")
for row in img:
    print("  " + "".join("#" if v > 8 else "+" if v > 3 else "." for v in row))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
clf = SVC(kernel="rbf", gamma=0.001, C=10).fit(X_tr, y_tr)
pred = clf.predict(X_te)
print(f"\nTest accuracy: {accuracy_score(y_te, pred):.3f} on {len(y_te)} unseen images")

# Confusion matrix (Chapter 9): row = true digit, column = predicted digit.
# Zeroing the diagonal leaves only the mistakes; unravel_index turns the
# position of the biggest one into a (row, column) pair.
cm = confusion_matrix(y_te, pred)
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"Most common mistake: true {i} predicted as {j} ({off[i, j]} time(s))")
print("Per-digit accuracy:", {d: round(float(cm[d, d] / cm[d].sum()), 2) for d in range(10)})

Images: 1797 | pixels per image: 64 | classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

Image 0 (label 0):
  ..+##...
  ..####+.
  ..#..#+.
  .+#..++.
  .++..#+.
  .+#..#+.
  ..#+##..
  ..+##...

Test accuracy: 0.994 on 540 unseen images
Most common mistake: true 3 predicted as 7 (1 time(s))
Per-digit accuracy: {0: 1.0, 1: 1.0, 2: 1.0, 3: 0.98, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 0.98, 9: 0.98}


## Example 2: Image Retrieval by Example

This is query-by-example. We pick eight images as queries and ask the system for the five most similar images in the collection, using cosine similarity between pixel vectors. Precision@5 is the share of those five with the same digit as the query. Most queries score a perfect 1.0. One messy '3' brings back two '8's, which shows that similar-looking digits are the hard cases. The last line shows query-by-semantics: because every image is labeled, we can ask for 'sevens' by keyword. Cosine similarity compares the direction of two vectors: it is their dot product after each is scaled to length 1, so 1.0 means the same direction. NearestNeighbors reports cosine distance, which is 1 minus the similarity, so the code computes 1 − dist to get the similarity back.

In [2]:
# Chapter 2 - Example 2: Content-based image retrieval, query-by-example (QBE)
# Give the system one picture; it returns the most similar pictures in the collection.
import numpy as np
from sklearn.datasets import load_digits
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

digits = load_digits()
X, y = digits.data, digits.target
rng = np.random.default_rng(21)
query_ids = rng.choice(len(X), 8, replace=False)
db_mask = np.ones(len(X), bool); db_mask[query_ids] = False   # queries are not in the database
X_db, y_db = X[db_mask], y[db_mask]

# Cosine similarity: compare the 'direction' of the pixel vectors
index = NearestNeighbors(n_neighbors=5, metric="cosine").fit(normalize(X_db))

precisions = []
for q in query_ids:
    dist, idx = index.kneighbors(normalize(X[q:q + 1]))
    found = y_db[idx[0]]
    p_at_5 = np.mean(found == y[q])
    precisions.append(p_at_5)
    sims = np.round(1 - dist[0], 3)
    print(f"Query digit {y[q]} -> top-5 labels {found.tolist()}  similarity {sims.tolist()}  "
          f"precision@5 = {p_at_5:.1f}")
print(f"\nMean precision@5 over {len(query_ids)} queries: {np.mean(precisions):.2f}")

# Query-by-semantics (QBS) needs labels/annotations instead of an example picture:
want = 7
print(f"\nQBS 'show me sevens': {np.sum(y_db == want)} annotated images match the keyword '{want}'")

Query digit 5 -> top-5 labels [5, 5, 5, 5, 5]  similarity [0.966, 0.922, 0.906, 0.875, 0.873]  precision@5 = 1.0
Query digit 0 -> top-5 labels [0, 0, 0, 0, 0]  similarity [0.981, 0.979, 0.973, 0.973, 0.973]  precision@5 = 1.0
Query digit 4 -> top-5 labels [4, 4, 4, 4, 4]  similarity [0.987, 0.968, 0.963, 0.949, 0.948]  precision@5 = 1.0
Query digit 3 -> top-5 labels [3, 8, 3, 8, 3]  similarity [0.931, 0.913, 0.907, 0.897, 0.895]  precision@5 = 0.6
Query digit 0 -> top-5 labels [0, 0, 0, 0, 0]  similarity [0.944, 0.943, 0.933, 0.932, 0.928]  precision@5 = 1.0
Query digit 3 -> top-5 labels [3, 3, 3, 3, 3]  similarity [0.974, 0.966, 0.958, 0.955, 0.953]  precision@5 = 1.0
Query digit 3 -> top-5 labels [3, 3, 3, 3, 3]  similarity [0.962, 0.949, 0.947, 0.938, 0.933]  precision@5 = 1.0
Query digit 3 -> top-5 labels [3, 3, 3, 3, 3]  similarity [0.935, 0.92, 0.908, 0.903, 0.903]  precision@5 = 1.0

Mean precision@5 over 8 queries: 0.95

QBS 'show me sevens': 179 annotated images match the keyw

## Example 3: Fraud Detection in Banking

Banks see millions of normal transactions and very few frauds, often with no fraud labels at all. An Isolation Forest learns what normal looks like and flags anything unusual. Here, 20 frauds (large amounts, at night, far from home, many purchases in an hour) are hidden among 2,000 normal transactions. The model flags 21 transactions and catches all 20 frauds, with only one false alarm. We use the true labels only at the end, to check the result. Precision (20 of 21 = 0.95) is the share of flagged transactions that really are fraud, and recall (20 of 20 = 1.00) is the share of frauds caught; Chapter 9 covers both. One caution: contamination=0.01 tells the model to flag the most unusual 1% of transactions, which is very close to the true fraud rate. That is part of why the result is so clean. In real life you would not know the rate and would have to estimate it.

In [3]:
# Chapter 2 - Example 3: AI in banking - spotting fraudulent transactions (anomaly detection)
# No fraud labels are used for training: Isolation Forest learns what 'normal' looks like.
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score

rng = np.random.default_rng(0)
n_normal, n_fraud = 2000, 20
# Features: amount ($), hour of day, distance from home (km), purchases in last hour
normal = np.column_stack([rng.lognormal(3.5, 0.6, n_normal), rng.normal(14, 3, n_normal) % 24,
                          rng.exponential(5, n_normal), rng.poisson(1, n_normal)])
fraud = np.column_stack([rng.lognormal(6.5, 0.5, n_fraud), rng.normal(3, 1, n_fraud) % 24,
                         rng.exponential(400, n_fraud), rng.poisson(6, n_fraud)])
X = np.vstack([normal, fraud])
y = np.r_[np.zeros(n_normal), np.ones(n_fraud)]          # used ONLY to check the result

# contamination = share of points to flag (1%). In real use this is a guess from
# domain knowledge; here it is close to the true fraud rate, which helps a lot.
model = IsolationForest(n_estimators=200, contamination=0.01, random_state=0).fit(X)
flag = (model.predict(X) == -1).astype(int)              # -1 means 'anomaly'
score = -model.score_samples(X)                          # higher = more unusual

print(f"Transactions: {len(X)}  (true frauds hidden inside: {n_fraud})")
print(f"Flagged as suspicious: {flag.sum()}")
print(f"Precision: {precision_score(y, flag):.2f}   Recall: {recall_score(y, flag):.2f}")

print("\nTop 5 most unusual transactions:")
print("  amount($)  hour  distance(km)  buys/hr  truly fraud?")
for i in np.argsort(score)[::-1][:5]:
    a, h, d, b = X[i]
    print(f"  {a:9.0f}  {h:4.0f}  {d:12.0f}  {b:7.0f}  {'yes' if y[i] else 'no'}")

Transactions: 2020  (true frauds hidden inside: 20)
Flagged as suspicious: 21
Precision: 0.95   Recall: 1.00

Top 5 most unusual transactions:
  amount($)  hour  distance(km)  buys/hr  truly fraud?
        959     5           775        7  yes
       1041     3          1536        5  yes
        557     3           711       10  yes
        918     4           482        8  yes
        822     4           576        6  yes
